# Step 6: Post-process Recordings

**Input:** `recordings/speaker_XX/OOV/*.wav` and `recordings/speaker_XX/IV/*.wav`  
**Output:** `recordings_processed/speaker_XX/OOV/*.wav` and `IV/*.wav`  

**What this does (in order):**
1. Verify all expected files exist (223 per speaker)
2. Check audio specs (sample rate, channels, bit depth)
3. Trim leading/trailing silence
4. Normalize peak level to -3 dBFS
5. Export as 48kHz, 16-bit, mono WAV
6. Quality report — duration stats, SNR estimate, any flagged files

## Install dependencies

In [2]:
# Run once
!pip install -q pydub soundfile numpy tqdm
# Linux / Colab only — skip if already installed
!apt-get install -y -q ffmpeg 2>/dev/null || echo 'ffmpeg already installed or skipped'

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


## Imports & Config

In [ ]:
#unzipping since previous resluts were zipped if not zipped u can skip this step part
import zipfile
from pathlib import Path

ZIP_FILE = "recordings.zip"
EXTRACT_TO = "."

with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_TO)

print("Extraction complete.")

# Verify structure
recordings_path = Path("recordings")
if recordings_path.exists():
    print(f"Found: {recordings_path}")
    print(f"Items inside: {len(list(recordings_path.iterdir()))}")
else:
    print("Warning: 'recordings' folder not found after extraction.")

Extraction complete.
Found: recordings
Items inside: 1


In [ ]:
import os
import json
import shutil
import numpy as np
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import detect_leading_silence
from tqdm import tqdm
import soundfile as sf

# Config 
RAW_DIR       = Path('recordings')             # raw recordings root
OUT_DIR       = Path('recordings_processed')   # post-processed output root
BACKUP_DIR    = Path('recordings_raw_backup')  # safety backup

SPEAKERS      = ['speaker_01']   # add more: ['speaker_01', 'speaker_02', ...] if there are multiple speakers
SECTIONS      = ['OOV', 'IV']

# Expected counts (from Step 5)
EXPECTED = {'OOV': 119, 'IV': 104}    # adjust if yours differ

# Audio processing
TARGET_SR     = 48000    # Hz
TARGET_DBFS   = -3.0     # peak normalization target
SILENCE_THRESH = -40     # dBFS — below this = silence
SILENCE_PAD   = 300      # ms of silence to keep on each side after trim
MIN_DUR_MS    = 500      # flag files shorter than this
MAX_DUR_MS    = 15000    # flag files longer than this
MIN_SNR_DB    = 20       # flag files below this estimated SNR

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config OK')
print(f'  Speakers    : {SPEAKERS}')
print(f'  Raw dir     : {RAW_DIR}')
print(f'  Output dir  : {OUT_DIR}')

Config OK
  Speakers    : ['speaker_01']
  Raw dir     : recordings
  Output dir  : recordings_processed


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## Backup raw recordings (run once)

In [6]:
if BACKUP_DIR.exists():
    print(f'Backup already exists at {BACKUP_DIR} — skipping.')
else:
    print(f'Copying {RAW_DIR} → {BACKUP_DIR} ...')
    shutil.copytree(RAW_DIR, BACKUP_DIR)
    print(f'Backup complete.')

Copying recordings → recordings_raw_backup ...
Backup complete.


## Verify all expected files exist

In [7]:
print('Verifying file counts...\n')
all_ok = True
missing_files = []

for spk in SPEAKERS:
    print(f'── {spk} ──')
    for section in SECTIONS:
        section_dir = RAW_DIR / spk / section
        if not section_dir.exists():
            print(f'  ✗  {section_dir} does not exist')
            all_ok = False
            continue

        expected_n = EXPECTED[section]
        found_files = sorted(section_dir.glob('*.wav'))
        found_ids   = {f.stem for f in found_files}

        # Check for expected IDs
        expected_ids = {f'{section}_{i:04d}' for i in range(1, expected_n + 1)}
        missing = expected_ids - found_ids
        extra   = found_ids - expected_ids

        status = '✓' if not missing else '✗'
        print(f'  {status}  {section}: found {len(found_files)}/{expected_n} files')
        if missing:
            for m in sorted(missing):
                print(f'       MISSING: {m}.wav')
                missing_files.append(f'{spk}/{section}/{m}.wav')
            all_ok = False
        if extra:
            for e in sorted(extra):
                print(f'       EXTRA  : {e}.wav  (unexpected — check naming)')

print()
if all_ok:
    print('✅ All expected files present. Ready to process.')
else:
    print(f'⚠️  {len(missing_files)} file(s) missing — resolve before continuing.')

Verifying file counts...

── speaker_01 ──
  ✓  OOV: found 119/119 files
  ✓  IV: found 104/104 files

✅ All expected files present. Ready to process.


## Post-processing functions

In [ ]:
def estimate_snr(audio: AudioSegment) -> float:
#  Rough SNR estimate: signal power vs noise floor (first 200ms).
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
    if len(samples) == 0:
        return 0.0
    noise_samples = samples[:int(audio.frame_rate * 0.2)]  # first 200ms
    signal_rms = np.sqrt(np.mean(samples ** 2)) + 1e-9
    noise_rms  = np.sqrt(np.mean(noise_samples ** 2)) + 1e-9
    return float(20 * np.log10(signal_rms / noise_rms))


def postprocess(in_path: Path, out_path: Path,
                target_dbfs=TARGET_DBFS,
                silence_thresh=SILENCE_THRESH,
                pad_ms=SILENCE_PAD,
                target_sr=TARGET_SR) -> dict:
#   Load → trim silence → normalize → resample → export.
    audio = AudioSegment.from_wav(str(in_path))
    raw_dur_ms = len(audio)
    raw_dbfs   = audio.max_dBFS

    # Trim leading silence
    start_trim = detect_leading_silence(audio, silence_threshold=silence_thresh)
    # Trim trailing silence (reverse trick)
    end_trim   = detect_leading_silence(audio.reverse(), silence_threshold=silence_thresh)
    end_pos    = len(audio) - end_trim

    # Safety: don't trim more than 90% of the file
    if start_trim > end_pos or (end_pos - start_trim) < raw_dur_ms * 0.1:
        start_trim, end_pos = 0, len(audio)  # leave as-is

    audio = audio[start_trim:end_pos]

    # Add padding on both sides
    pad = AudioSegment.silent(duration=pad_ms, frame_rate=audio.frame_rate)
    audio = pad + audio + pad

    # Estimate SNR before normalization
    snr = estimate_snr(audio)

    # Normalize peak
    if audio.max_dBFS < -60:  # essentially silent — flag it
        change = 0
    else:
        change = target_dbfs - audio.max_dBFS
    audio = audio.apply_gain(change)

    # Force 48kHz mono 16-bit
    audio = audio.set_frame_rate(target_sr).set_channels(1).set_sample_width(2)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    audio.export(str(out_path), format='wav')

    return {
        'file'        : out_path.name,
        'raw_dur_ms'  : raw_dur_ms,
        'proc_dur_ms' : len(audio),
        'raw_dbfs'    : round(raw_dbfs, 1),
        'snr_db'      : round(snr, 1),
        'trimmed_ms'  : (raw_dur_ms - (end_pos - start_trim)),
        'ok'          : True,
    }

print('Functions defined OK')

Functions defined OK


##  Run post-processing

In [9]:
all_results = []   # flat list of metadata dicts
flagged     = []   # files needing manual review

for spk in SPEAKERS:
    print(f'\n══ Processing {spk} ══')
    for section in SECTIONS:
        in_dir  = RAW_DIR / spk / section
        out_dir = OUT_DIR / spk / section
        out_dir.mkdir(parents=True, exist_ok=True)

        wav_files = sorted(in_dir.glob('*.wav'))
        if not wav_files:
            print(f'  ⚠  No WAV files in {in_dir}')
            continue

        print(f'  {section}: {len(wav_files)} files')
        for wav in tqdm(wav_files, desc=f'  {section}', leave=False):
            out_path = out_dir / wav.name
            try:
                meta = postprocess(wav, out_path)
                meta['speaker'] = spk
                meta['section'] = section

                # Flag suspicious files
                flags = []
                if meta['proc_dur_ms'] < MIN_DUR_MS:
                    flags.append(f'too short ({meta["proc_dur_ms"]}ms)')
                if meta['proc_dur_ms'] > MAX_DUR_MS:
                    flags.append(f'too long ({meta["proc_dur_ms"]}ms)')
                if meta['snr_db'] < MIN_SNR_DB:
                    flags.append(f'low SNR ({meta["snr_db"]}dB)')
                if meta['raw_dbfs'] < -30:
                    flags.append(f'very quiet input ({meta["raw_dbfs"]}dBFS)')

                meta['flags'] = ' | '.join(flags)
                if flags:
                    flagged.append(meta)

                all_results.append(meta)

            except Exception as e:
                err = {'speaker': spk, 'section': section,
                       'file': wav.name, 'flags': f'ERROR: {e}', 'ok': False}
                all_results.append(err)
                flagged.append(err)
                print(f'  ✗  {wav.name}: {e}')

print(f'\nDone. Processed {len(all_results)} files total.')
print(f'Flagged for review: {len(flagged)}')


══ Processing speaker_01 ══
  OOV: 119 files


  IV: 104 files



Done. Processed 223 files total.
Flagged for review: 0


## Quality report

In [12]:
import statistics

ok_results = [r for r in all_results if r.get('ok', False)]
durations  = [r['proc_dur_ms'] for r in ok_results]
snrs       = [r['snr_db']      for r in ok_results]

print('=' * 55)
print('  POST-PROCESSING QUALITY REPORT')
print('=' * 55)
print(f'  Files processed     : {len(all_results)}')
print(f'  Errors              : {len([r for r in all_results if not r.get("ok", True)])}')
print(f'  Flagged for review  : {len(flagged)}')
print()

if durations:
    print(f'  Duration (processed)')
    print(f'    Min   : {min(durations)/1000:.1f}s')
    print(f'    Max   : {max(durations)/1000:.1f}s')
    print(f'    Mean  : {statistics.mean(durations)/1000:.1f}s')
    print(f'    Median: {statistics.median(durations)/1000:.1f}s')
    print()

if snrs:
    print(f'  Estimated SNR')
    print(f'    Min   : {min(snrs):.1f} dB')
    print(f'    Max   : {max(snrs):.1f} dB')
    print(f'    Mean  : {statistics.mean(snrs):.1f} dB')
    print()

if flagged:
    print(f'  ⚠  FILES NEEDING MANUAL REVIEW ({len(flagged)}):')
    print(f'  {"File":<20s} {"Speaker":<12s} {"Section":<8s} {"Issue"}')
    print('  ' + '-' * 52)
    for r in flagged:
        print(f'  {r["file"]:<20s} {r.get("speaker","?"):<12s} '
              f'{r.get("section","?"):<8s} {r.get("flags","?")}')
else:
    print('  ✅ No files flagged.')

print('=' * 55)

  POST-PROCESSING QUALITY REPORT
  Files processed     : 223
  Errors              : 0
  Flagged for review  : 0

  Duration (processed)
    Min   : 6.5s
    Max   : 11.8s
    Mean  : 8.9s
    Median: 8.7s

  Estimated SNR
    Min   : 222.0 dB
    Max   : 260.5 dB
    Mean  : 247.2 dB

  ✅ No files flagged.


## Save QC report to CSV

In [13]:
import csv

report_path = OUT_DIR / 'qc_report.csv'
fieldnames  = ['speaker', 'section', 'file', 'raw_dur_ms', 'proc_dur_ms',
                'raw_dbfs', 'snr_db', 'trimmed_ms', 'flags', 'ok']

with open(report_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(all_results)

print(f'QC report saved → {report_path}')

# Also save flagged-only
if flagged:
    flagged_path = OUT_DIR / 'qc_flagged.csv'
    with open(flagged_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(flagged)
    print(f'Flagged report saved → {flagged_path}')
    print(f'\nReview flagged files and re-record if needed, then re-run Cells 6–8.')
else:
    print('No flagged files — ready for Step 7.')

QC report saved → recordings_processed/qc_report.csv
No flagged files — ready for Step 7.


## Verify processed output structure

In [14]:
print('Processed output structure:\n')
total_files = 0
all_verified = True

for spk in SPEAKERS:
    print(f'  {spk}/')
    for section in SECTIONS:
        out_dir    = OUT_DIR / spk / section
        found      = list(out_dir.glob('*.wav'))
        expected_n = EXPECTED[section]
        ok         = len(found) == expected_n
        status     = '✓' if ok else '✗'
        print(f'    {status} {section}/  {len(found)}/{expected_n} files')
        total_files += len(found)
        if not ok:
            all_verified = False

print(f'\n  Total processed files : {total_files}')
print()
if all_verified:
    print('✅ All files verified. Proceed to Step 7 — Intelligibility Evaluation.')
else:
    print('⚠️  File count mismatch — check flagged files and re-record if needed.')

Processed output structure:

  speaker_01/
    ✓ OOV/  119/119 files
    ✓ IV/  104/104 files

  Total processed files : 223

✅ All files verified. Proceed to Step 7 — Intelligibility Evaluation.


## Final summary

In [15]:
print('=' * 55)
print('  STEP 6 COMPLETE')
print('=' * 55)
print(f'  Raw backup      : {BACKUP_DIR}')
print(f'  Processed audio : {OUT_DIR}')
print(f'  QC report       : {OUT_DIR}/qc_report.csv')
print()
print('  Next step → Step 7: Intelligibility Evaluation')
print('  - Run your TTS model on evaluation_sentences.csv')
print('  - Have raters score each synthesized word')
print('  - Compute IER% per category (OOV vs IV)')
print('=' * 55)

  STEP 6 COMPLETE
  Raw backup      : recordings_raw_backup
  Processed audio : recordings_processed
  QC report       : recordings_processed/qc_report.csv

  Next step → Step 7: Intelligibility Evaluation
  - Run your TTS model on evaluation_sentences.csv
  - Have raters score each synthesized word
  - Compute IER% per category (OOV vs IV)


In [26]:
import shutil

folder_path = "/content/recordings_processed"
output_zip = "/content/recordings_processed.zip"

shutil.make_archive(output_zip.replace(".zip", ""), 'zip', folder_path)

print("Zipped successfully:", output_zip)

Zipped successfully: /content/recordings_processed.zip


In [27]:
from google.colab import files

files.download("/content/recordings_processed.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>